# AlphaGWAS Quick Start Tutorial

This notebook demonstrates how to use AlphaGWAS for variant prioritization at GWAS loci.

## What you'll learn:
1. Loading and exploring GWAS summary statistics
2. Configuring the pipeline
3. Running variant prioritization
4. Visualizing results
5. Interpreting top variants

## Setup

First, let's import the required libraries and set up the environment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add parent directory to path
sys.path.insert(0, '..')

# Import AlphaGWAS modules
from scripts import extract_variants, score_variants, utils
from scripts.visualize import create_manhattan_plot, create_tissue_heatmap

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete!')

## 1. Generate Sample GWAS Data

For this tutorial, we'll generate synthetic GWAS data. In practice, you would load your own summary statistics.

In [ ]:
# Generate sample GWAS data
np.random.seed(42)
n_variants = 1000

gwas_data = pd.DataFrame({
    'chromosome': np.random.choice([str(i) for i in range(1, 23)], n_variants),
    'position': np.random.randint(1_000_000, 200_000_000, n_variants),
    'rsid': [f'rs{np.random.randint(1000, 9999999)}' for _ in range(n_variants)],
    'effect_allele': np.random.choice(['A', 'C', 'G', 'T'], n_variants),
    'other_allele': np.random.choice(['A', 'C', 'G', 'T'], n_variants),
    'beta': np.random.normal(0, 0.1, n_variants),
    'se': np.abs(np.random.normal(0.02, 0.005, n_variants)),
    'pvalue': np.concatenate([
        np.random.uniform(1e-15, 1e-8, 50),    # Significant
        np.random.uniform(1e-5, 1, n_variants - 50)  # Not significant
    ])
})

print(f'Generated {len(gwas_data)} variants')
print(f'Significant variants (p < 5e-8): {(gwas_data["pvalue"] < 5e-8).sum()}')
gwas_data.head()

## 2. Explore the Data

In [ ]:
# Summary statistics
print('Data Summary:')
print(f'  Total variants: {len(gwas_data):,}')
print(f'  Chromosomes: {gwas_data["chromosome"].nunique()}')
print(f'  P-value range: {gwas_data["pvalue"].min():.2e} - {gwas_data["pvalue"].max():.2f}')
print(f'  Beta range: {gwas_data["beta"].min():.3f} - {gwas_data["beta"].max():.3f}')

In [ ]:
# P-value distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram of -log10(p)
axes[0].hist(-np.log10(gwas_data['pvalue']), bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(x=-np.log10(5e-8), color='red', linestyle='--', label='p = 5e-8')
axes[0].set_xlabel('-log10(p-value)')
axes[0].set_ylabel('Count')
axes[0].set_title('P-value Distribution')
axes[0].legend()

# QQ plot
observed = -np.log10(np.sort(gwas_data['pvalue']))
expected = -np.log10(np.linspace(1/len(observed), 1, len(observed)))
axes[1].scatter(expected, observed, alpha=0.5, s=10)
axes[1].plot([0, max(expected)], [0, max(expected)], 'r--')
axes[1].set_xlabel('Expected -log10(p)')
axes[1].set_ylabel('Observed -log10(p)')
axes[1].set_title('QQ Plot')

plt.tight_layout()
plt.show()

## 3. Extract Significant Variants

In [ ]:
# Extract genome-wide significant variants
pval_threshold = 5e-8
significant = gwas_data[gwas_data['pvalue'] < pval_threshold].copy()

print(f'Found {len(significant)} significant variants')
significant.sort_values('pvalue').head(10)

## 4. Simulate AlphaGenome Predictions

In a real analysis, these would come from the AlphaGenome API. Here we simulate predictions.

In [ ]:
# Simulate predictions
tissues = ['Heart_Left_Ventricle', 'Whole_Blood', 'Liver', 'Brain_Cortex']
modalities = ['expression', 'chromatin_accessibility']

predictions = []
for _, variant in significant.iterrows():
    variant_id = f"chr{variant['chromosome']}:{variant['position']}"
    for tissue in tissues:
        for modality in modalities:
            # Simulate effect correlated with -log10(p)
            base_effect = -np.log10(variant['pvalue']) / 15
            predictions.append({
                'variant_id': variant_id,
                'rsid': variant['rsid'],
                'tissue': tissue,
                'modality': modality,
                'effect_size': abs(np.random.normal(base_effect, 0.1)),
                'confidence': np.random.uniform(0.6, 1.0),
                'prediction': np.random.normal(0, base_effect)
            })

predictions_df = pd.DataFrame(predictions)
print(f'Generated {len(predictions_df)} predictions')
predictions_df.head()

## 5. Score and Rank Variants

In [ ]:
# Initialize scorer
scorer = score_variants.VariantScorer({})

# Calculate tissue scores
tissue_scores = scorer.calculate_tissue_scores(predictions_df)
print(f'Tissue scores shape: {tissue_scores.shape}')

# Calculate consensus scores
consensus_scores = scorer.calculate_consensus_scores(tissue_scores)
print(f'Consensus scores shape: {consensus_scores.shape}')

# Rank variants
ranked_variants = scorer.rank_variants(consensus_scores)
print(f'\nTop 10 variants:')
ranked_variants[['variant_id', 'consensus_score', 'max_effect', 'final_rank']].head(10)

## 6. Visualize Results

In [ ]:
# Extract chromosome and position from variant_id for plotting
ranked_variants[['chromosome', 'position']] = ranked_variants['variant_id'].str.extract(r'chr(\w+):(\d+)')
ranked_variants['position'] = pd.to_numeric(ranked_variants['position'])

# Manhattan-style plot
fig = create_manhattan_plot(ranked_variants, score_col='consensus_score', highlight_top_n=10)
plt.show()

In [ ]:
# Tissue heatmap
fig = create_tissue_heatmap(tissue_scores, top_n_variants=15)
plt.show()

In [ ]:
# Score distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Consensus score distribution
axes[0].hist(ranked_variants['consensus_score'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Consensus Score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Consensus Scores')

# Score vs Rank
axes[1].scatter(ranked_variants['final_rank'], ranked_variants['consensus_score'], alpha=0.6)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Consensus Score')
axes[1].set_title('Score vs Rank')

plt.tight_layout()
plt.show()

## 7. Interpret Results

In [ ]:
# Top variants summary
print('='*60)
print('TOP 10 PRIORITIZED VARIANTS')
print('='*60)

for i, row in ranked_variants.head(10).iterrows():
    print(f"\nRank {int(row['final_rank'])}:")
    print(f"  Variant: {row['variant_id']}")
    print(f"  Consensus Score: {row['consensus_score']:.4f}")
    print(f"  Max Effect: {row['max_effect']:.4f}")
    print(f"  Significant Tissues: {int(row['n_significant_tissues'])}")

## 8. Export Results

In [ ]:
# Save to files
output_dir = Path('../data/output')
output_dir.mkdir(parents=True, exist_ok=True)

ranked_variants.to_csv(output_dir / 'tutorial_ranked_variants.tsv', sep='\t', index=False)
tissue_scores.to_csv(output_dir / 'tutorial_tissue_scores.tsv', sep='\t', index=False)

print(f'Results saved to {output_dir}')

## Next Steps

Now that you've completed the tutorial, you can:

1. **Use real GWAS data**: Replace the simulated data with your own summary statistics
2. **Configure AlphaGenome**: Set up API access for real predictions
3. **Add annotations**: Use the annotation module for ClinVar, gnomAD, GTEx data
4. **Run fine-mapping**: Integrate SuSiE/FINEMAP results
5. **Pathway analysis**: Run enrichment analysis on top variant genes

See the full documentation at: https://glbala87.github.io/alphagwas